Evan Edelstein
EN.605.645.82.SP26

# Module 8 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

In [ ]:
from copy import deepcopy
import random
import json
from math import log2, inf
from typing import List, Dict, Set, Tuple

## Decision Trees

For this assignment you will be implementing and evaluating a Decision Tree using the ID3 Algorithm (**no** pruning or normalized information gain). Use the provided pseudocode. The data is located at (copy link):

http://archive.ics.uci.edu/ml/datasets/Mushroom

**Just in case** the UCI repository is down, which happens from time to time, I have included the data and name files on Canvas.

<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Important</strong>
    <p>
        No Pandas. The only acceptable libraries in this class are those contained in the `environment.yml`. No OOP, either. You can used Dicts, NamedTuples, etc. as your abstract data type (ADT) for the the tree and nodes.
    </p>
</div>

One of the things we did not talk about in the lectures was how to deal with missing values. There are two aspects of the problem here. What do we do with missing values in the training data? What do we do with missing values when doing classifcation?

There are a lot of different ways that we can handle this.
A common algorithm is to use something like kNN to impute the missing values.
We can use conditional probability as well.
There are also clever modifications to the Decision Tree algorithm itself that one can make.

We're going to do something simpler, given the size of the data set: remove the observations with missing values ("?").

You must implement the following functions:

`train` takes training_data and returns the Decision Tree as a data structure.

```
def train(training_data):
   # returns the Decision Tree.
```

`classify` takes a tree produced from the function above and applies it to labeled data (like the test set) or unlabeled data (like some new data).

```
def classify(tree, observations):
    # returns a list of classifications
```

`evaluate` takes a data set with labels (like the training set or test set) and the classification result and calculates the classification error rate:

$$error\_rate=\frac{errors}{n}$$

Do not use anything else as evaluation metric or the submission will be deemed incomplete, ie, an "F". (Hint: accuracy rate is not the error rate!).

`cross_validate` takes the data and uses 10 fold cross validation (from Module 3!) to `train`, `classify`, and `evaluate`. **Remember to shuffle your data before you create your folds**. I leave the exact signature of `cross_validate` to you but you should write it so that you can use it with *any* `classify` function of the same form (using higher order functions and partial application).

Following Module 3's material (course notes), `cross_validate` should print out a table in exactly the same format. What you are looking for here is a consistent evaluation metric cross the folds. Print the error rate to 4 decimal places. **Do not convert to a percentage.**

```
def pretty_print_tree(tree):
    # pretty prints the tree
```

This should be a text representation of a decision tree trained on the entire data set (no train/test).

To summarize...

Apply the Decision Tree algorithm to the Mushroom data set using 10 fold cross validation and the error rate as the evaluation metric. When you are done, apply the Decision Tree algorithm to the entire data set and print out the resulting tree.

**Note** Because this assignment has a natural recursive implementation, you should consider using `deepcopy` at the appropriate places.


### Provided Functions

You do not need to document these.

You can use this function to read the data file.

In [ ]:
def parse_data(file_name: str) -> list[list]:
    data = []
    file = open(file_name, "r")
    for line in file:
        datum = line.rstrip().split(",")
        data.append(datum)
    random.shuffle(data)
    return data

You can use this function to create 10 folds for 5x2 cross validation.

In [3]:
def create_folds(xs: list, n: int) -> list[list[list]]:
    k, m = divmod(len(xs), n)
    # be careful of generators...
    return list(xs[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(n))

Put your code after this line:

-----

In [ ]:
def create_node(attribute):
    return {"node" : attribute, "children": []}

In [ ]:
def add_child(parent, child, value):
    parent["children"].append({"value": value, "child": child})
    return parent


In [ ]:
def get_children(node):
    return [(edge["value"], edge["child"]) for edge in node["children"]]

In [ ]:
def pretty_print_tree(root, tree):  # DFS
    rows = []
    frontier = [(root, [])]

    while frontier:
        parent, path = frontier.pop()
        children = get_children(parent)

        if not children:
            decision = []
            for feature, attr in path:
                decision.append(f"{feature['node']}: {attr} ->")
            decision.append(f"| {parent['node']} |")
            rows.append(" ".join(decision))

        for attr, child in children:
            child_path = path + [(parent, attr)]
            frontier.append((child, child_path))

    print("\n".join(rows))
    return

In [ ]:

def parse_attrs(filename):
    attr_map = {}
    attributes = {}
    with open(filename, "r") as fh:
        data = json.load(fh)
    for feature, attr in data.items():
        attr_map[feature] = {}
        attributes[feature] = []
        for single_letter, name in attr.items():
            attributes[feature].append(name)
            attr_map[feature][single_letter] = name
    return attributes, attr_map

In [ ]:


def rename_data(data, attributes, attr_map):
    return [[attr_map[attr][i] for i, attr in zip(row, attributes)] for row in data]


In [ ]:
def is_homogeneous(data, label_index):
    if len(set(row[label_index] for row in data)) == 1:
        return True
    return False


In [ ]:
def get_first_label(data, label_index) -> str:
    return data[0][label_index]

In [ ]:

def get_majority_label(data, label_index):
    counts = {}
    for row in data:
        label = row[label_index]
        if label in counts:
            counts[label] += 1
        else:
            counts[label] = 1

    return max(counts, key=lambda k: counts[k])

In [ ]:
def calculate_entropy(data, attr_index, attr, attributes, label_index, labels):
    subset_sizes = {i: 0 for i in attributes[attr]}
    label_counts = {l: {a: 0 for a in attributes[attr]} for l in labels}
    total_size = 0

    for row in data:
        observation = row[attr_index]
        label = row[label_index]
        subset_sizes[observation] += 1
        label_counts[label][observation] += 1
        total_size += 1

    entropy = 0
    for feature in attributes[attr]:
        subset_size = subset_sizes[feature]
        subset_entropy = 0
        for label in labels:
            p = label_counts[label][feature]
            if subset_size == 0:
                continue
            p = p / subset_size
            if p <= 0.0:
                continue
            subset_entropy += -1 * p * log2(p)
        entropy += (subset_size / total_size) * subset_entropy
    return entropy

In [ ]:

def pick_best_attribute(data, features, attributes, label_index, labels, trace=False):
    best_attr = None
    best_entropy = inf
    best_index = -1

    for index, attr in features.items():
        entropy = calculate_entropy(data, index, attr, attributes, label_index, labels)

        if entropy < best_entropy:
            best_attr = attr
            best_entropy = entropy
            best_index = index
    if trace: 
        print(f"Lowest Entropy Attr: {best_index=} {best_attr} {best_entropy=}")

    return best_index, best_attr


In [ ]:
def domain(attributes, feature):
    return attributes[feature]


In [ ]:
def subset_data(data, attr_index, attr):
    return [deepcopy(row) for row in data if row[attr_index] == attr]

In [ ]:
def remove_features(features, attr_index):
    new_features = deepcopy(features)
    new_features.pop(attr_index)
    return new_features

In [ ]:
def split_features(attributes, label):
    features = {}
    labels = None
    label_index = -1
    for index, attr in enumerate(attributes):
        if attr == label:
            labels = attributes[attr]
            label_index = index
        else:
            features[index] = attr
    return features,labels,label_index

In [ ]:
def id3(data, features, attributes, label_index, labels, default_label=None, trace=False):
    if trace:
        print(f"[DEBUG] {features=}, {attributes=}")

    if len(data) == 0:
        print("[DEBUG] Base Case - empty data") 
        default_label = get_majority_label(data, label_index) if default_label is None else default_label
        return create_node(default_label)

    if is_homogeneous(data, label_index):
        print("[DEBUG] Base Case - homogenous data")
        return create_node(get_first_label(data, label_index))

    if len(features) == 0:
        print("[DEBUG] Base Case - empty features")
        return create_node(get_majority_label(data, label_index))
    
    index, attr = pick_best_attribute(data, features, attributes, label_index, labels)

    node = create_node(attr)
    default_label = get_majority_label(data, label_index)
    
    for value in domain(attributes, attr):
        subset = subset_data(data, index, value)
        # print(f"Partitioning {value=} from {attr=}")
        new_features = remove_features(features, index)
        child = id3(subset, new_features, attributes, label_index, labels, default_label, trace)
        node = add_child(node, child, value)
    return node

In [ ]:
def train(data, attributes, label, trace= False):
    features, labels, label_index = split_features(attributes, label)
    tree = id3(data, features, attributes, label_index, labels, None, trace)
    return tree

## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.